---
authors:
  - name: Emma Pidduck
    url: https://www.linkedin.com/in/emmapidduck/
---

# Day 20 · Hexagons — snow on the Alps

<img src="assets/ECMWF_Core_Logo_RGB.png" width="170" alt="ECMWF logo">

### #30DayMapChallenge 2026 · ECMWF IFS 9 km open data · Map by Emma Pidduck

> ⚠️ **Just for fun — not an official ECMWF product.** Claude (Anthropic) helped build
> these notebooks. They are **not** official ECMWF notebooks and are **not** supported by
> ECMWF — a personal [#30DayMapChallenge](https://30daymapchallenge.com/) experiment.

A companion to [Day 20 · Italy, rain by the hour](day20_hexagons_italy): the same 3D hexagons, this time
for **snowfall** over the **Alps**. ECMWF's 9 km forecast is gathered into **H3 hexagons** about 17 km
across, kept where they fall inside the Alpine arc, stood up as **3D columns** and played through all
ten days of the forecast:

- **`MODE = "stack"`** — each column **grows** as the snow piles up, ending on the ten-day total.
- **`MODE = "bounce"`** — each column jumps with the **snow falling right now** (mm per hour).
- **`MODE = "both"`** — columns **grow** with the total and **flash** while it is snowing on them.

Snowfall is in **millimetres of water** (what the model accumulates). As a rule of thumb, 1 mm of
water makes about **1 cm of fresh snow**, but the ratio runs from about 5:1 for wet snow to 20:1 for
powder.

Self-contained: method, code, output, data licence and tools in one place, runnable on any
laptop with `pip`.

## Method

- Fetch **snowfall** (`sf`, metres of water equivalent) from the latest IFS run at **0.1°** (9 km)
  for every output step to +240 h. Open data has 3-hourly steps to +144 h, then 6-hourly.
- `sf` is accumulated from the start of the run. Differencing consecutive steps gives the
  **snowfall rate** in each 3 h or 6 h block (divided by its length, so both are mm/h); the rate is
  then eased between block mid-points so the columns move smoothly instead of strobing.
- Bin every grid point into an **H3 hexagon** (res 5) and keep those whose centre is inside the
  Natural Earth outline of the **Alps** (Nice to Vienna). The surrounding land is drawn flat for context. Each hexagon becomes a prism, drawn far-to-near on an
  equal-area plane tilted 35° — a hand-rolled 3D view, plain matplotlib.
- Height and colour both show the value; a QR code in the corner points back to this notebook.

## Data

| Dataset | What is used | Licence |
|---|---|---|
| [ECMWF Open Data](https://www.ecmwf.int/en/forecasts/datasets/open-data) — IFS HRES (`oper`), 0.1° | snowfall `sf`, latest 00/12 UTC run, +0 to +240 h | CC-BY-4.0 |
| [Natural Earth](https://www.naturalearthdata.com) 1:10m | Alps outline (`geography_regions_polys`), land | public domain |

No account needed. The download is about 65 global fields (~40 MB); they are cached in `data/`.

## Licence & attribution

Access to ECMWF Open Data is governed by the **Creative Commons CC-BY-4.0** licence and the
associated [Terms of Use](https://apps.ecmwf.int/datasets/licences/general/).

> In applying this licence, ECMWF does not waive the privileges and immunities granted to it
> by virtue of its status as an intergovernmental organisation.

When you share a map made from this data, attribute it:
`© ECMWF · Source: www.ecmwf.int · Licence: CC-BY-4.0 and ECMWF Terms of Use`

## Tools

| Tool | Role |
|---|---|
| [`ecmwf-opendata`](https://github.com/ecmwf/ecmwf-opendata) | retrieve the IFS forecast (`resol="0p1"` for the 9 km grid) |
| `xarray` · `cfgrib` | read the GRIB |
| [`h3`](https://h3geo.org) | hexagonal binning |
| `cartopy` · `shapely` | Natural Earth shapes, the equal-area plane |
| `matplotlib` · `qrcode` · `ffmpeg` | render, the QR code, the MP4 (and GIF) |

```bash
pip install ecmwf-opendata xarray cfgrib h3 cartopy shapely matplotlib qrcode pillow
```
`ffmpeg` must be on your path for the video (`brew install ffmpeg` / `apt install ffmpeg`).

In [ ]:
import os, shutil, subprocess
from datetime import timedelta
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib.image as mpimg
import matplotlib.cm as mcm
import matplotlib.ticker
from matplotlib.collections import PolyCollection
from matplotlib.colors import PowerNorm
import cartopy.crs as ccrs
import cartopy.io.shapereader as shpreader
from shapely.geometry import Point
from shapely.ops import unary_union
from shapely.prepared import prep
import h3, qrcode
import xarray as xr
from ecmwf.opendata import Client

plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Avenir Next", "Helvetica Neue", "Arial", "DejaVu Sans"]

### Configuration

`RUN = None` takes the **latest** run, so re-running the notebook makes today's animation.
Pin one with `RUN = "2026111900"`.

In [ ]:
MODE   = "stack"           # "stack" = snow so far (mm water) · "bounce" = snow right now (mm/h) · "both" = height so far, colour now
RUN    = None              # None = latest; or "YYYYMMDDHH"
REGION = "ALPS"            # Natural Earth geography_regions_polys NAME
RES, EL = 5, 35.0          # H3 resolution (~17 km hexagons) · camera elevation (degrees)
FPH, FPS = 2, 24           # frames per forecast hour · video frame rate -> 240 h in 20 s
DPI = 100                  # 12 in wide -> 1200 px; height follows the map
URL = "https://github.com/piddy376/30dmc2026/blob/main/day20_hexagons_alps.ipynb"
OUT = f"day20_hexagons_alps_{MODE}"
THEME  = "light"           # "light" (white page, wet = dark blue) or "dark" (navy, columns glow)
if THEME == "dark":
    BG, INK, MUT, LAND, ARC, COAST, EDGE = "#0b1d33", "#f2f5f8", "#9fb0c3", "#102843", "#173556", "#2c4a6b", "#0b1d33"
else:
    BG, INK, MUT, LAND, ARC, COAST, EDGE = "#ffffff", "#222222", "#6b6b6b", "#f6f7f9", "#e6e9ef", "#111111", "#ffffff"
HALO = [pe.withStroke(linewidth=2.6, foreground=BG)]
PLACES = {"Grenoble": (45.19, 5.72), "Chamonix": (45.92, 6.87), "Zermatt": (46.02, 7.75),
          "St. Moritz": (46.50, 9.84), "Innsbruck": (47.27, 11.39), "Cortina": (46.54, 12.14),
          "Bad Gastein": (47.12, 13.13)}

## Retrieve every step of the run

In [ ]:
client = Client("ecmwf")
STEPS = list(range(0, 145, 3)) + list(range(150, 241, 6))
run = client.latest(stream="oper", type="fc", step=240, param="sf", resol="0p1") if RUN is None \
      else __import__("datetime").datetime.strptime(RUN, "%Y%m%d%H")
grib = f"data/day20_alps_sf_0p1_{run:%Y%m%d%H}.grib2"
os.makedirs("data", exist_ok=True)
if not os.path.exists(grib):
    client.retrieve(date=run.strftime("%Y%m%d"), time=run.hour, step=STEPS, stream="oper", type="fc",
                    levtype="sfc", param="sf", resol="0p1", target=grib)
ds = xr.open_dataset(grib, engine="cfgrib")["sf"]                  # [step, latitude, longitude], metres
steps = (ds.step.values / np.timedelta64(1, "h")).astype(int)
LAT1 = ds.latitude.values; LON1 = ((ds.longitude.values + 180) % 360) - 180
print(f"run {run:%Y-%m-%d %H} UTC · {len(steps)} steps {steps[0]}..{steps[-1]} h")

## The Alps, binned into hexagons

In [ ]:
rec = next(r for r in shpreader.Reader(shpreader.natural_earth("10m", "physical", "geography_regions_polys")).records()
           if r.attributes["NAME"] == REGION)
geom = rec.geometry; w, s, e, n = geom.bounds; pg = prep(geom)
parts = list(geom.geoms) if hasattr(geom, "geoms") else [geom]
# flat land around the arc for context: every country touching the box, merged (no borders drawn)
from shapely.geometry import box
BOX = box(w - 3.0, s - 2.0, e + 3.0, n + 6.0)                   # past every edge of the frame
land = unary_union([r.geometry for r in shpreader.Reader(shpreader.natural_earth("10m", "cultural", "admin_0_countries")).records()
                    if r.geometry.intersects(BOX)]).intersection(BOX)
LANDPARTS = list(land.geoms) if hasattr(land, "geoms") else [land]
COASTS = [c.intersection(BOX) for c in shpreader.Reader(shpreader.natural_earth("10m", "physical", "coastline")).geometries()
          if c.intersects(BOX)]
COASTS = [g for c in COASTS for g in (c.geoms if hasattr(c, "geoms") else [c]) if not g.is_empty]

ml = (LON1 >= w - 0.2) & (LON1 <= e + 0.2); mla = (LAT1 >= s - 0.2) & (LAT1 <= n + 0.2)
lon, lat = np.meshgrid(LON1[ml], LAT1[mla])
pt_cell = np.array([h3.latlng_to_cell(float(a), float(o), RES) for a, o in zip(lat.ravel(), lon.ravel())])
cells = [c for c in np.unique(pt_cell) if pg.contains(Point(h3.cell_to_latlng(c)[::-1]))]
idx = {c: i for i, c in enumerate(cells)}
which = np.array([idx.get(c, -1) for c in pt_cell]); keep = which >= 0
count = np.bincount(which[keep], minlength=len(cells))

# accumulated tp (mm) per hexagon, at every step: shape [step, hexagon]
sub = ds.values[:, mla][:, :, ml].reshape(len(steps), -1) * 1000.0
acc = np.array([np.bincount(which[keep], f[keep], len(cells)) / count for f in sub])
acc = np.maximum.accumulate(np.maximum(acc, 0), axis=0)            # tidy tiny negative packing noise (m -> mm water above)
print(f"{len(cells)} hexagons · 10-day max {acc[-1].max():.0f} mm")

## Frames: snow right now, or snow so far

Rate in each block = Δ`tp` / Δt, placed at the block's mid-point and eased (linearly) between
mid-points. The running total is `sf` itself, interpolated in time.

In [ ]:
t = np.arange(0, steps[-1] * FPH + 1) / FPH                        # forecast hour of every frame
mid = (steps[1:] + steps[:-1]) / 2
rate_blk = np.diff(acc, axis=0) / np.diff(steps)[:, None]           # mm/h, [block, hexagon]
rate = np.stack([np.interp(t, mid, rate_blk[:, i]) for i in range(len(cells))], axis=1)
total = np.stack([np.interp(t, steps, acc[:, i]) for i in range(len(cells))], axis=1)

if MODE == "bounce":
    VAL, VMAX, GAMMA, UNIT = rate, max(2.0, float(np.percentile(rate.max(axis=0), 99))), 0.5, "mm/h"
    LABEL = "Snow falling now (mm of water per hour)"
else:
    VAL, VMAX, GAMMA, UNIT = total, max(20.0, float(np.percentile(total[-1], 99.5))), 0.6, "mm"
    LABEL = "Snow so far (mm of water ≈ cm of fresh snow)"
SNOW = ["#cfd8ec", "#9fb3dd", "#7a86c9", "#6a56b0", "#55308f", "#3a145f"]   # ice blue -> violet -> deep indigo
cmap = (matplotlib.colors.LinearSegmentedColormap.from_list("frost", ["#2a3f6b", "#5b78b8", "#9fb7e6", "#dbe6fb", "#ffffff"])
        if THEME == "dark" else matplotlib.colors.LinearSegmentedColormap.from_list("snow", SNOW)); norm = PowerNorm(GAMMA, vmin=0, vmax=VMAX, clip=True)
DRY = 0.05 if MODE == "bounce" else 0.5                             # mm/h or mm: below this, no column
CVAL, cnorm = VAL, norm                                            # what the colour shows
if MODE == "both":                                                  # height = so far, colour = right now
    CVAL = rate; cnorm = PowerNorm(0.5, vmin=0, vmax=max(2.0, float(np.percentile(rate.max(axis=0), 99))), clip=True)
    LABEL = "Snow falling now (mm of water per hour)"

def colour(c, h):
    "Colour of one column: the snowfall rate, never paler than a quiet tint that deepens with the total so far."
    if MODE != "both": return np.array(cmap(cnorm(c))[:3])
    return np.array(cmap(max(cnorm(c), 0.10 + 0.25 * norm(h)))[:3])
print(f"{len(t)} frames · colour top {VMAX:.1f} {UNIT}")

## The 3D stage: an equal-area plane, tilted toward the camera

In [ ]:
cen = geom.centroid
laea = ccrs.LambertAzimuthalEqualArea(central_longitude=cen.x, central_latitude=cen.y)
sin_el, cos_el = np.sin(np.radians(EL)), np.cos(np.radians(EL))

def ground(lons, lats):
    p = laea.transform_points(ccrs.PlateCarree(), np.asarray(lons, float), np.asarray(lats, float))
    return p[:, 0] / 1000.0, p[:, 1] / 1000.0                       # km, camera looks north

def screen(gx, gy, z=0.0):
    return np.column_stack([gx, gy * sin_el + z * cos_el])

HEX = []
for c in cells:
    b = h3.cell_to_boundary(c); gx, gy = ground([p[1] for p in b], [p[0] for p in b]); HEX.append((gx, gy))
order = np.argsort([-gy.mean() for _, gy in HEX])                   # far columns first (painter's order)
span_km = max(np.ptp(np.concatenate([g[0] for g in HEX])), np.ptp(np.concatenate([g[1] for g in HEX])))
KM = 0.22 * span_km / VMAX                                          # tallest column = 22% of the arc's length

def prisms(v, cv):
    "Side faces + tops of every column: heights from v, colours from cv (one per hexagon)."
    polys, cols = [], []
    for i in order:
        if v[i] < DRY: continue                                     # dry: leave the ground (and coastline) showing
        gx, gy = HEX[i]; z = min(v[i], VMAX * 1.15) * KM; rgb = colour(cv[i], v[i])
        cx, cy = gx.mean(), gy.mean(); m = len(gx)
        if z > 0.05:
            for a in range(m):
                b = (a + 1) % m
                nx, ny = (gx[a] + gx[b]) / 2 - cx, (gy[a] + gy[b]) / 2 - cy
                if ny >= 0: continue                                # faces away from the camera
                polys.append(np.vstack([screen(gx[[a, b]], gy[[a, b]]), screen(gx[[b, a]], gy[[b, a]], z)]))
                cols.append(tuple(rgb * (0.62 + 0.22 * (nx / np.hypot(nx, ny) + 1) / 2)))
        polys.append(screen(gx, gy, z)); cols.append(tuple(rgb))
    return polys, cols

CITY = []
for name, (la, lo) in PLACES.items():
    near = [idx[k] for k in h3.grid_disk(h3.latlng_to_cell(la, lo, RES), 1) if k in idx]
    if near: CITY.append((name, *ground([lo], [la]), near))

## One frame — and the poster

In [ ]:
allx = np.concatenate([g[0] for g in HEX]); ally = np.concatenate([g[1] for g in HEX])
x0, x1 = allx.min() - 0.04 * span_km, allx.max() + 0.04 * span_km
y0 = ally.min() * sin_el - 0.12 * span_km    # room below the arc for the clock
# top of the page = the highest any column ever reaches (or a city label), not a worst case
peak = np.minimum(VAL.max(axis=0), VMAX * 1.15) * KM * cos_el
y1 = max(max(gy.max() * sin_el + p for (_, gy), p in zip(HEX, peak)),
         max(gy[0] * sin_el + peak[near].max() for _, _, gy, near in CITY) + 0.10 * span_km) + 0.01 * span_km
W, TOP, BOT = 12.0, 1.30, 1.00                                       # inches: page width, title band, footer band
MAPH = W * (y1 - y0) / (x1 - x0); H = TOP + MAPH + BOT              # the map spans the full width

qr = qrcode.QRCode(border=1, box_size=10, error_correction=qrcode.constants.ERROR_CORRECT_M)
qr.add_data(URL); qr.make(fit=True)
QR = np.array(qr.make_image(fill_color="#0b1d33", back_color="white").convert("RGB"))
LOGO = mpimg.imread("assets/ECMWF_Core_Logo_WHITE.png" if THEME == "dark" else "assets/ECMWF_Core_Logo_RGB.png")

def stage():
    fig = plt.figure(figsize=(W, H), dpi=DPI); fig.patch.set_facecolor(BG)
    ax = fig.add_axes([0, BOT / H, 1, MAPH / H]); ax.set_facecolor(BG); ax.axis("off")
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
    for g in LANDPARTS:                                             # neighbouring land, flat and quiet
        ex, ey = ground(*g.exterior.coords.xy)
        ax.fill(*screen(ex, ey).T, facecolor=LAND, edgecolor="none", zorder=0.4)
    for g in COASTS:                                                # real coastline only (box edges stay invisible)
        ex, ey = ground(*g.coords.xy)
        ax.plot(*screen(ex, ey).T, color=MUT, lw=0.5, zorder=0.5)
    for g in parts:                                                 # the Alpine arc
        ex, ey = ground(*g.exterior.coords.xy)
        ax.fill(*screen(ex, ey).T, facecolor=ARC, edgecolor=COAST, linewidth=1.1 if THEME == "light" else 0.6, zorder=1)
    fy = lambda inch_from_top: 1 - inch_from_top / H
    fig.text(0.035, fy(0.30), {"bounce": "Snow on the Alps, hour by hour", "stack": "Ten days of snow, piling up on the Alps",
              "both": "Snow on the Alps, flashing and piling up"}[MODE],
             fontsize=28, fontweight="bold", color=INK, va="top")
    fig.text(0.035, fy(0.88), "ECMWF's 9 km forecast in hexagons about 17 km across · " +
             {"bounce": "columns jump with the snow falling right now",
              "stack": "columns grow as the snow adds up",
              "both": "columns grow with the total and flash while it snows"}[MODE], fontsize=13, color=MUT, va="top")
    lx = fig.add_axes([0.80, fy(0.95), 0.165, 0.60 / H]); lx.imshow(LOGO); lx.axis("off")
    # time, bottom-left over the Po valley side
    clock = ax.text(0.02, 0.07, "", transform=ax.transAxes, fontsize=22, fontweight="bold", color=INK, va="bottom")
    sub = ax.text(0.02, 0.055, "", transform=ax.transAxes, fontsize=11, color=MUT, va="top")
    q = 0.70 / H
    qx = fig.add_axes([0.965 - q * H / W, 0.10 / H, q * H / W, q]); qx.imshow(QR, interpolation="nearest"); qx.axis("off")
    fig.text(0.955 - q * H / W, 0.45 / H, "Notebook\n& code", fontsize=9, color=MUT, ha="right", va="center", linespacing=1.3)
    cax = fig.add_axes([0.035, 0.58 / H, 0.30, 0.13 / H])
    cb = fig.colorbar(mcm.ScalarMappable(norm=cnorm, cmap=cmap), cax=cax, orientation="horizontal", extend="max")
    tk = [x for x in matplotlib.ticker.MaxNLocator(5).tick_values(0, cnorm.vmax) if x <= cnorm.vmax * 0.92]
    cb.set_ticks(tk); cb.ax.set_title(LABEL, color=INK, fontsize=10, loc="left", pad=5); cb.outline.set_visible(False)
    cb.ax.tick_params(colors=MUT, labelsize=9)
    fig.text(0.035, 0.08 / H, "© ECMWF · IFS 9 km open data · CC BY 4.0 · #30DayMapChallenge Day 20 · Emma Pidduck · "
             "not an official ECMWF product", fontsize=8.5, color=MUT, va="bottom")
    return fig, ax, clock, sub

def draw(fig, ax, clock, sub, k, dyn):
    for a in dyn: a.remove()
    dyn.clear(); v = VAL[k]
    polys, cols = prisms(v, CVAL[k])
    dyn.append(ax.add_collection(PolyCollection(polys, facecolors=cols, edgecolors=EDGE, linewidths=0.15, zorder=2)))
    lift = 0.05 * span_km
    for name, gx, gy, near in CITY:
        top = screen(gx, gy, min(v[near].max(), VMAX * 1.15) * KM)[0]
        dyn.append(ax.plot([top[0]] * 2, [top[1], top[1] + lift], color=INK, lw=0.8, zorder=5)[0])
        dyn.append(ax.text(top[0], top[1] + lift, f"{name}\n{total[k, near].mean():.0f} mm", ha="center", va="bottom",
                           fontsize=11, fontweight="bold", color=INK, zorder=6, linespacing=1.1, path_effects=HALO))
    vt = run + timedelta(hours=float(t[k]))
    clock.set_text(f"{vt:%a %-d %b · %H:00} UTC")
    sub.set_text(f"Run {run:%-d %b %Y %H} UTC · +{t[k]:.0f} h · labels: total so far")

fig, ax, clock, sub = stage(); dyn = []
k_poster = int(np.argmax(VAL.sum(axis=1))) if MODE == "bounce" else (
    int(np.argmax(rate.sum(axis=1))) if MODE == "both" else len(t) - 1)
draw(fig, ax, clock, sub, k_poster, dyn)
fig.savefig(f"{OUT}.png", facecolor=BG)
plt.show() if matplotlib.get_backend().lower() != "agg" else None

![Poster frame](day20_hexagons_alps_stack.png)

## The animation — frames piped straight into ffmpeg

In [ ]:
fig.canvas.draw(); wpx, hpx = fig.canvas.get_width_height(); wpx, hpx = wpx // 2 * 2, hpx // 2 * 2
ff = shutil.which("ffmpeg")
assert ff, "ffmpeg not found — install it for the video"
hold = FPS                                                          # one second still at each end
proc = subprocess.Popen([ff, "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "rgba",
                         "-s", f"{fig.canvas.get_width_height()[0]}x{fig.canvas.get_width_height()[1]}",
                         "-r", str(FPS), "-i", "-", "-vf", f"crop={wpx}:{hpx}:0:0", "-c:v", "libx264",
                         "-pix_fmt", "yuv420p", "-crf", "20", "-movflags", "+faststart", f"{OUT}.mp4"],
                        stdin=subprocess.PIPE)
seq = [0] * hold + list(range(len(t))) + [len(t) - 1] * hold
for j, k in enumerate(seq):
    if j == 0 or k != seq[j - 1]:
        draw(fig, ax, clock, sub, k, dyn); fig.canvas.draw()
    proc.stdin.write(bytes(fig.canvas.buffer_rgba()))
proc.stdin.close(); proc.wait(); plt.close(fig)

# a lighter GIF for social: half size, 12 fps
subprocess.run([ff, "-y", "-loglevel", "error", "-i", f"{OUT}.mp4", "-vf",
                "fps=10,scale=540:-1:flags=lanczos,split[a][b];[a]palettegen=max_colors=128[p];[b][p]paletteuse=dither=bayer",
                f"{OUT}.gif"], check=True)
print(f"wrote {OUT}.mp4, {OUT}.gif ({len(seq) / FPS:.0f} s)")

![Ten days of IFS 9 km snowfall over the Alps as 3D hexagon columns growing with the total](day20_hexagons_alps_stack.gif)

*Crisper MP4: [`day20_hexagons_alps_stack.mp4`](day20_hexagons_alps_stack.mp4) · `MODE = "both"` for the
flashing version ([`day20_hexagons_alps_both.mp4`](day20_hexagons_alps_both.mp4)) · ECMWF IFS open data (CC-BY-4.0).*

## Note

The view is a hand-built projection, not a true 3D renderer: columns are drawn far to near, which is
exact for prisms standing on a plane seen from one side. The camera looks north; swap the sign of
`gy` in `ground()` to look south. Any other Natural Earth range works by changing `REGION` and `PLACES`
(try `"PYRENEES"`).